# 00 — Start Here: Setup and Data

**Who this is for.** A physics graduate student joining the QML-for-LArTPC project. You know quantum
mechanics and HEP data analysis; no machine-learning background is assumed. Every ML term is
explained when it first appears.

**The notebook series:**

| Notebook | What it does |
|---|---|
| **00 (this one)** | checks your environment, builds the training dataset, verifies it |
| **01** | looks at the data — what the model will actually see |
| **02** | the quantum circuit — what it is, proof it is correct, how fast it runs |

Deeper written references, linked throughout: [`../docs/PRIMER.md`](../docs/PRIMER.md) (concepts,
textbook-style) and [`../docs/RUNBOOK.md`](../docs/RUNBOOK.md) (shell-level setup: installing
packages, GPU drivers — things that cannot run inside a notebook).

**One habit to adopt from day one:** before trusting any result from a notebook, do
**Kernel → Restart & Run All**. Notebooks let you run cells out of order, which silently leaves the
kernel in a state no one can reproduce. Restart-and-run-all is the only defence.

## What this project is, in one paragraph

A liquid argon TPC records charged particles as 2D images (wire number × drift time, intensity =
ionization charge). Reconstructing a neutrino interaction starts with deciding, for each deposit of
charge, what made it: a narrow trail from a muon/pion/proton (**track**) or a spreading
electromagnetic cascade from an electron/photon (**shower**). We test whether a classifier built
partly from a small *quantum circuit* can do this as well as a conventional neural network while
using **far fewer adjustable parameters** — and, if it can, whether the cause is *quantum* or merely
*small*.

**The one idea to hold onto:** a classifier is a function `f(x; θ)` — image in, score out — with a
list of adjustable numbers θ (the **parameters**). *Training* is nothing more exotic than a fit:

| the χ² fit you already know | training a network |
|---|---|
| 2–5 parameters | 100 – 100,000 parameters |
| minimise χ² | minimise a **loss function** |
| MINUIT / `curve_fit` | **gradient descent** |
| judge by χ²/dof | judge on **held-out data** |

Keep that analogy in mind and most ML vocabulary becomes familiar. More: [PRIMER §1](../docs/PRIMER.md#1-what-this-project-computes).

## Step 1 — check the environment

The cell below imports every package the project needs and prints its version. If any import fails,
the fix is a shell command, not a notebook cell — see [RUNBOOK §1](../docs/RUNBOOK.md#1-install),
which also explains two machine-specific traps (a tiny `/tmp` partition, and a GPU driver older than
pip's default torch).

In [1]:
# Find the project root so imports and file paths work no matter where
# jupyter was launched from. The notebooks live in <root>/notebooks/.
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))          # makes "import qmllar" work

# Import each dependency and report its version.
import numpy, h5py, sklearn, pennylane
print("numpy    ", numpy.__version__)       # array maths -- everything builds on this
print("h5py     ", h5py.__version__)        # reads the campaign HDF5 files
print("sklearn  ", sklearn.__version__)     # ROC/AUC computation, later
print("pennylane", pennylane.__version__)   # builds + simulates quantum circuits (the readable path)

# torch is the automatic-differentiation library: it computes dL/dtheta for us,
# which is what makes training possible at all. It also drives the GPU.
import torch
print("torch    ", torch.__version__, "| built for CUDA", torch.version.cuda)
print("GPU available:", torch.cuda.is_available(),
      "|", torch.cuda.device_count(), "device(s)")
if torch.cuda.is_available():
    print("          ", torch.cuda.get_device_name(0))

numpy     2.4.3
h5py      3.16.0
sklearn   1.9.0
pennylane 0.45.1


torch     2.13.0+cu126 | built for CUDA 12.6
GPU available: True | 2 device(s)
           NVIDIA GeForce RTX 4090


### ✅ What you should see

All five imports succeed, and — the critical line — **`GPU available: True` with 2 devices**
(two RTX 4090s on this machine).

**If you see `GPU available: False` but `2 device(s)`:** that is the known trap — a torch build for
CUDA 13 on a driver that supports 12.5. It installs cleanly and fails only at run time. Fix in
[RUNBOOK §6](../docs/RUNBOOK.md#6-troubleshooting). You can still run notebooks 00–02 without the
GPU; training (later) is what needs it.

## Step 2 — find the campaign data

The training data comes from the Wire-Cell AI/ML campaign: ~400,000 simulated DUNE far-detector
neutrino events **with per-pixel truth labels** (plus ~2M without labels, used for other purposes).
Full inventory: [`../docs/wc-aiml-dataset-inventory.md`](../docs/wc-aiml-dataset-inventory.md).

The dataset has already moved disk once, so the code *resolves* its location rather than hardcoding
it (override with the environment variable `WC_AIML_BASE` if it moves again).

In [2]:
import os

# The RAW campaign data (492 GB) lives on the BNL machine wcgpu1. It is only
# needed to BUILD the patch file -- if data/patches_31.npz is already present
# (e.g. copied to your FNAL area), everything from Step 3 onward works without it.
try:
    from qmllar import extract
    BASE = extract.BASE                     # resolves lazily; raises if data absent
    print("campaign data:", BASE)
    for sample in ("fdhd_sparse_nue_truthlabel", "fdhd_sparse_numu_truthlabel"):
        n = len(os.listdir(os.path.join(BASE, sample, "001")))
        print(f"  {sample}/001 : {n} archives  (~{n*10:,} events)")
except FileNotFoundError:
    BASE = None
    print("Raw campaign data NOT on this machine -- that is fine as long as")
    print("data/patches_31.npz exists (next cell checks). To point at a copy:")
    print("  export WC_AIML_BASE=/path/to/training_samples")

campaign data: /data/0/jjo/WC_AIML/training_samples
  fdhd_sparse_nue_truthlabel/001 : 1044 archives  (~10,440 events)
  fdhd_sparse_numu_truthlabel/001 : 1031 archives  (~10,310 events)


### ✅ What you should see

The resolved path (currently `/data/0/jjo/WC_AIML/training_samples`) and roughly **1,000 archives ≈
10,000 events** per flavour in batch `001`. Batches `001`–`010` exist per flavour, so ~100k labelled
events each — we use one batch and it is already plenty.

Two facts about this data worth knowing *before* touching it (both explained in notebook 01):
`nue` = electron-neutrino events, `numu` = muon-neutrino; and the per-pixel truth has one dangerous
convention (negative track IDs) that fails silently if mishandled.

## Step 3 — build the training dataset

`scripts/build_patches.py` turns raw campaign archives into one training-ready file. For every
labelled pixel it keeps, it:

1. cuts a **31×31 window** of the charge image centred on that pixel;
2. derives the label *in memory*: pixel → dominant particle's track ID → `abs()` → PDG code →
   track or shower (the campaign files are **never modified**);
3. assigns the patch to train/validation/test **by archive**, never individually — patches from the
   same event overlap in content, so splitting them across sets would leak information (notebook 01
   and [PRIMER §3.3](../docs/PRIMER.md#33-three-splits-and-the-leakage-trap) explain this trap).

The build takes **~4 minutes** (the data sits on a spinning disk; it was 55 s when it lived on an
SSD). The cell skips the build if the output file already exists — delete the file to force a
rebuild.

In [3]:
import os
out = ROOT / "data" / "patches_31.npz"

if out.exists():
    # Already built (or copied here) -- a rebuild with the same seed is identical.
    print(f"already built: {out}  ({out.stat().st_size/1e6:.0f} MB) -- skipping")
elif BASE is None:
    print("No patch file AND no raw campaign data on this machine.")
    print("Get data/patches_31.npz (201 MB) from someone with wcgpu1 access --")
    print("it is the only file the rest of the notebook series needs.")
else:
    # The leading "!" hands the line to the shell -- how a notebook runs a script.
    # --workers 16: modest parallelism; more readers on a spinning disk fight each other.
    !cd {ROOT} && python3 scripts/build_patches.py --n-archives 800 --workers 16

already built: /home/jjo/Work/QML/data/patches_31.npz  (201 MB) -- skipping


### ✅ What you should see (from the build, or from the original build log)

```
done; 0 archive errors
  415509 patches, shape (31, 31), 201 MB -> data/patches_31.npz
  train:  292603 patches  track=0.425 shower=0.575  archives=1120
  val  :   60677 patches  track=0.395 shower=0.605  archives=240
  test :   62229 patches  track=0.433 shower=0.567  archives=240
```

Three numbers to register:

- **415,509 patches.** The published study we benchmark against trained on **100–1000**. We have
  ~300× their data from a fraction of one batch — the ability to extend their result to realistic
  data volumes is one of this project's main cards.
- **track ≈ 0.42 / shower ≈ 0.58** — a nearly balanced two-class problem (why that matters, and why
  it required pooling both neutrino flavours: notebook 01).
- **archives=1120/240/240** — the split is by archive. The next step checks that no archive leaks
  across splits.

## Step 4 — verify before trusting

Three checks, and **all three must pass before any training is ever run**. Each guards against a
failure that produces *plausible-looking but wrong* results — the dangerous kind.

In [4]:
import numpy as np

d = np.load(ROOT / "data" / "patches_31.npz")
patches, cls, efrac = d["patches"], d["cls"], d["efrac"]
split, archive = d["split"], d["archive"]

# --- Check 1: no archive appears in more than one split -------------------
# If an event's patches landed in both train and test, the model could score
# well by recognising *that event* rather than learning physics ("leakage").
by_archive = {a: set(split[archive == a]) for a in np.unique(archive)}
n_leaky = sum(len(s) > 1 for s in by_archive.values())
print(f"archives spanning >1 split : {n_leaky}   (must be 0)")

# --- Check 2: every patch has a complete label-quality value --------------
# efrac = the labelling particle's share of the energy at the centre pixel.
# A NaN here would mean the coordinate join between truth and image failed.
print(f"efrac NaN fraction         : {np.isnan(efrac).mean()}   (must be 0.0)")

# --- Check 3: the labelled pixel is really at the window centre -----------
# Every patch is supposed to be centred on the pixel that was labelled, and
# a labelled pixel by definition carries charge.
centred = np.mean(patches[:, 15, 15] > 0)
print(f"centre pixel carries charge: {centred:.1%}  (must be 100%)")

# Summary table per split
print()
for name, v in (("train", 0), ("val", 1), ("test", 2)):
    m = split == v
    print(f"  {name:5s} {m.sum():7,} patches   track {np.mean(cls[m]==0):.3f}  "
          f"shower {np.mean(cls[m]==1):.3f}")

archives spanning >1 split : 0   (must be 0)
efrac NaN fraction         : 0.0   (must be 0.0)
centre pixel carries charge: 100.0%  (must be 100%)

  train 292,603 patches   track 0.425  shower 0.575
  val    60,677 patches   track 0.395  shower 0.605
  test   62,229 patches   track 0.433  shower 0.567


### ✅ What you should see

`0`, `0.0`, and `100.0%`. Anything else means **stop** — do not train on this file:

- *Leaky archives > 0* → the split logic broke; any test score would be inflated memorisation.
- *NaN efrac > 0* → the truth-to-image coordinate join failed for some pixels.
- *Centre < 100%* → the extraction mis-centred windows; the labels would describe the wrong pixel.

None of these would crash a training job. All of them would quietly corrupt its results — which is
exactly why they are checked here, once, before anything else.

---

## Where to next

→ **Notebook 01** — look at the patches themselves: what tracks and showers actually look like,
what an ambiguous label is, and the one data trap everyone must know about.